# Timestamp: 2026-05-11 17:04:42

# 따릉이 1년 야간 재배치 KPI 비교 Notebook

이 노트북은 `NightReallocationOperationsOntology`를 활용한 **ontology-semantic-flow**와 종래 방식에 해당하는 **native baseline**을 같은 날짜, 같은 capacity, 같은 station 후보 조건에서 비교한다.

- 야간 재배치 window: `23:00~05:00`
- 다음날 부족 진단 window: `07:00~10:00`
- 기본 capacity: `200명 * 3 trips/night * 1 bike = 600 bikes/night`
- 거리 비용: `branch_data.branch_x/branch_y` 위경도 기반 Haversine distance
- Markov transition: `rent_data.branchnum_r -> rent_data.branchnum_b`

기본 실행은 1년 전체를 대상으로 한다. 빠른 검토만 하려면 설정 셀에서 `max_days = 30`처럼 제한하면 된다.


## KPI 정의

| KPI | 의미 | 해석 |
| :--- | :--- | :--- |
| `morning_shortage_reduction` | 추천 이동 후 오전 부족 위험 감소율 | 높을수록 좋음 |
| `avg_distance_km` | 추천 route 평균 거리 | 낮을수록 현장 이동 부담이 작음 |
| `p95_distance_km` | route 거리 p95 | 긴 이동 꼬리를 통제하는 지표 |
| `bike_moves_per_km` | 이동 거리 대비 보충 대수 효율 | 높을수록 좋음 |
| `capacity_utilization` | 600 bikes/night capacity 사용률 | 너무 낮으면 보수적, 너무 높으면 현장 부담 가능 |
| `recommendation_coverage` | 추천 row의 source/evidence/confidence 보유율 | ontology 관리 가능성 지표 |
| `weak_context_guard` | worker/route log 부재 항목이 weak-context/inferred로 표기된 비율 | 근거 과장 방지 지표 |
| `semantic_flow_uplift` | ontology-semantic-flow가 native 대비 개선한 정도 | 효과 판정 지표 |


In [ ]:
from __future__ import annotations

from datetime import date, datetime
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tools" / "scripts").exists():
    PROJECT_ROOT = Path("/home/user/Documents/01_Projects/01_Active/obybk")

TOOLS_SCRIPTS = PROJECT_ROOT / "tools" / "scripts"
if str(TOOLS_SCRIPTS) not in sys.path:
    sys.path.insert(0, str(TOOLS_SCRIPTS))

from rag.ttareungi_rag import processed_bike_cloud_dir
from rag.run_ttareungi_reallocation_simulation import (
    DEFAULT_CANDIDATE_LIMIT,
    DEFAULT_TOTAL_CAPACITY,
    POLICY_NATIVE,
    POLICY_SEMANTIC_FLOW,
    SimulationConfig,
    _after_state_rows,
    _candidate_pairs,
    _infer_latest_plan_date,
    _policy_metrics,
    _simulate_policy,
    build_markov_edges,
    build_station_states,
)

DATA_DIR = processed_bike_cloud_dir(PROJECT_ROOT)
OUTPUT_ROOT = PROJECT_ROOT / "data" / "processed" / "exports" / "reallocation_yearly_kpi_notebook_runs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("project_root:", PROJECT_ROOT)
print("data_dir:", DATA_DIR)
print("policy_native:", POLICY_NATIVE)
print("policy_semantic_flow:", POLICY_SEMANTIC_FLOW)


In [ ]:
# 실행 설정
analysis_year = 2024
candidate_limit = DEFAULT_CANDIDATE_LIMIT

# 1년 전체 실행 기본값이다. 빠른 확인만 하려면 30 같은 정수로 바꾼다.
max_days = None

# notebook 산출물 run directory
run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
run_stamp = datetime.strptime(run_timestamp, "%Y-%m-%d %H:%M:%S").strftime("%Y%m%d_%H%M%S")
run_dir = OUTPUT_ROOT / f"run_{run_stamp}_yearly_reallocation_kpi"
run_dir.mkdir(parents=True, exist_ok=True)

print("analysis_year:", analysis_year)
print("candidate_limit:", candidate_limit)
print("max_days:", max_days)
print("run_dir:", run_dir)


In [ ]:
def available_plan_dates(data_dir: Path, year: int) -> list[date]:
    """count_data의 date_rt 기준으로 해당 연도 plan_date 후보를 만든다."""
    count_path = data_dir / "count_data.parquet"
    frame = pd.read_parquet(count_path, columns=["date_rt"])
    dates = pd.to_datetime(frame["date_rt"], errors="coerce").dt.date.dropna()
    unique_dates = sorted({item for item in dates if item.year == year})
    return unique_dates


plan_dates = available_plan_dates(DATA_DIR, analysis_year)
if not plan_dates:
    latest = _infer_latest_plan_date(DATA_DIR)
    plan_dates = available_plan_dates(DATA_DIR, latest.year)
    analysis_year = latest.year

selected_plan_dates = plan_dates if max_days is None else plan_dates[:max_days]

print("available_dates:", len(plan_dates), plan_dates[0], "~", plan_dates[-1])
print("selected_dates:", len(selected_plan_dates), selected_plan_dates[0], "~", selected_plan_dates[-1])


In [ ]:
YEARLY_KPI_FIELDS = [
    "route_count",
    "total_recommended_bike_count",
    "morning_shortage_reduction",
    "avg_distance_km",
    "p95_distance_km",
    "bike_moves_per_km",
    "capacity_utilization",
    "recommendation_coverage",
    "weak_context_guard",
]


def simulate_one_plan_date(plan_date: date, candidate_limit: int) -> tuple[list[dict], list[dict]]:
    """하루치 native baseline과 ontology-semantic-flow KPI를 같은 조건에서 계산한다."""
    config = SimulationConfig(
        plan_date=plan_date,
        timestamp=f"{plan_date.isoformat()} 00:00:00",
        candidate_limit=candidate_limit,
    )
    states = build_station_states(DATA_DIR, config)
    markov_edges = build_markov_edges(DATA_DIR, config)
    pairs = _candidate_pairs(states, markov_edges, config)
    recommendations = (
        _simulate_policy(pairs, config, POLICY_NATIVE)
        + _simulate_policy(pairs, config, POLICY_SEMANTIC_FLOW)
    )
    state_rows = (
        _after_state_rows(states, recommendations, POLICY_NATIVE)
        + _after_state_rows(states, recommendations, POLICY_SEMANTIC_FLOW)
    )

    metric_rows: list[dict] = []
    for policy in [POLICY_NATIVE, POLICY_SEMANTIC_FLOW]:
        metrics = _policy_metrics(policy, recommendations, state_rows, config.total_capacity)
        metric_rows.append({"plan_date": plan_date.isoformat(), "policy": policy, **metrics})

    route_rows = [{"plan_date": plan_date.isoformat(), **row} for row in recommendations]
    return metric_rows, route_rows


def run_yearly_kpi_comparison(plan_dates: list[date], candidate_limit: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """1년 plan_date 목록에 대해 KPI와 route recommendation을 생성한다."""
    all_metrics: list[dict] = []
    all_routes: list[dict] = []
    failures: list[dict] = []

    for index, plan_date in enumerate(plan_dates, start=1):
        try:
            metric_rows, route_rows = simulate_one_plan_date(plan_date, candidate_limit)
            all_metrics.extend(metric_rows)
            all_routes.extend(route_rows)
        except Exception as exc:
            failures.append({"plan_date": plan_date.isoformat(), "error": type(exc).__name__, "message": str(exc)})
        if index == 1 or index % 10 == 0 or index == len(plan_dates):
            print(f"processed {index}/{len(plan_dates)}: {plan_date}")

    daily_kpis = pd.DataFrame(all_metrics)
    route_recommendations = pd.DataFrame(all_routes)
    if failures:
        pd.DataFrame(failures).to_csv(run_dir / "failures.csv", index=False)
    return daily_kpis, route_recommendations


In [ ]:
daily_kpis, route_recommendations = run_yearly_kpi_comparison(selected_plan_dates, candidate_limit)

daily_kpis.head()


In [ ]:
yearly_summary = (
    daily_kpis
    .groupby("policy", as_index=False)[YEARLY_KPI_FIELDS]
    .agg({
        "route_count": "mean",
        "total_recommended_bike_count": "mean",
        "morning_shortage_reduction": "mean",
        "avg_distance_km": "mean",
        "p95_distance_km": "mean",
        "bike_moves_per_km": "mean",
        "capacity_utilization": "mean",
        "recommendation_coverage": "mean",
        "weak_context_guard": "mean",
    })
    .round(4)
)

pivot = daily_kpis.pivot(index="plan_date", columns="policy", values="morning_shortage_reduction")
daily_uplift = (pivot[POLICY_SEMANTIC_FLOW] - pivot[POLICY_NATIVE]).rename("morning_shortage_reduction_delta")

coverage_pivot = daily_kpis.pivot(index="plan_date", columns="policy", values="recommendation_coverage")
coverage_uplift = (coverage_pivot[POLICY_SEMANTIC_FLOW] - coverage_pivot[POLICY_NATIVE]).rename("recommendation_coverage_delta")

semantic_flow_uplift = {
    "analysis_year": analysis_year,
    "evaluated_days": int(len(pivot)),
    "morning_shortage_reduction_delta_mean": round(float(daily_uplift.mean()), 4),
    "morning_shortage_reduction_delta_median": round(float(daily_uplift.median()), 4),
    "morning_shortage_reduction_delta_positive_rate": round(float((daily_uplift > 0).mean()), 4),
    "recommendation_coverage_delta_mean": round(float(coverage_uplift.mean()), 4),
}

yearly_summary


In [ ]:
semantic_flow_uplift


In [ ]:
import matplotlib.pyplot as plt

plot_frame = yearly_summary.set_index("policy")[[
    "morning_shortage_reduction",
    "avg_distance_km",
    "capacity_utilization",
    "recommendation_coverage",
]]

axes = plot_frame.T.plot(kind="bar", figsize=(12, 5), rot=25)
axes.set_title("Yearly KPI Comparison: native baseline vs ontology-semantic-flow")
axes.set_ylabel("metric value")
axes.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
daily_kpis.to_csv(run_dir / "daily_kpi_comparison.csv", index=False)
route_recommendations.to_csv(run_dir / "yearly_route_recommendations.csv", index=False)
yearly_summary.to_csv(run_dir / "yearly_summary.csv", index=False)
(run_dir / "semantic_flow_uplift.json").write_text(
    json.dumps(semantic_flow_uplift, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

manifest = {
    "timestamp": run_timestamp,
    "analysis_year": analysis_year,
    "selected_days": len(selected_plan_dates),
    "max_days": max_days,
    "candidate_limit": candidate_limit,
    "policies": [POLICY_NATIVE, POLICY_SEMANTIC_FLOW],
    "ontology": "NightReallocationOperationsOntology",
    "parent_ontology": "Ttareungi Domain Ontology-lite v2.1",
    "shift_window": "23:00~05:00",
    "evaluation_window": "07:00~10:00",
    "total_capacity": DEFAULT_TOTAL_CAPACITY,
    "outputs": [
        "daily_kpi_comparison.csv",
        "yearly_route_recommendations.csv",
        "yearly_summary.csv",
        "semantic_flow_uplift.json",
        "run_manifest.json",
    ],
}
(run_dir / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print("saved to", run_dir)


## 해석 기준

1. `morning_shortage_reduction_delta_mean > 0`이면 ontology-semantic-flow가 평균적으로 오전 부족 위험을 더 줄인 것이다.
2. `morning_shortage_reduction_delta_positive_rate`가 높을수록 특정 며칠이 아니라 연중 반복적으로 이득이 난 것이다.
3. ontology-semantic-flow의 `avg_distance_km` 또는 `p95_distance_km`가 너무 커지면 현장 이동 비용 증가로 해석한다.
4. `recommendation_coverage`와 `weak_context_guard`는 성능보다 **관리 가능성**과 **근거 과장 방지**를 보는 지표다.
5. 현재 데이터에는 실제 작업자 route log가 없으므로 200명 인력 조건은 `WorkforceCapacityConstraint` scenario이며, worker 관련 근거는 `weak-context`로 해석한다.
